# DTensor（Distributed Tensor，分布式张量）
- 是 PyTorch 2.0 引入的下一代分布式存储与计算抽象，旨在取代早期的 ShardedTensor。
- <font color='red'>它为大模型的分布式训练提供了统一的底层数据结构，使得开发者能够无缝构建张量并行（TP）、完全分片数据并行（FSDP）等复杂的并行策略。</font>

## DTensor 的核心设计理念
- 在传统的分布式训练中，由于<font color='red'>缺乏通用的抽象来在不同的并行策略之间架起桥梁</font>，导致各种解决方案互操作性差且难以使用。而无论是 Megatron-LM 还是 Colossal-AI 中的张量并行方案，往往都是基于 Transformer 架构定制的，不具备通用性。作为深度学习框架，<font color='red'>PyTorch 从更加通用的层面进行了设计，引入了 DTensor 作为基本构建块。</font>

- DTensor 提供了一系列原语（primitives）来表达张量的分布状态，主要包括<font color='red'>切片（Sharding）</font>和<font color='red'>重复（Replication）。</font>
  - <font color='red'>通过这一抽象，用户只需指明模型参数和输入的放置方式，</font>
   - <font color='green'>其他中间张量的放置方式会自动生成，无需显式调用底层的通信算子（如 NCCL）</font>，从而大幅降低了编写分布式代码的心智负担。

## DTensor 的主要用途
1. <b>统一检查点管理</b>：提供在 checkpointing 期间保存/加载 state_dict 的统一方法，即使存在极其复杂的张量存储分配策略（例如将 TP 与 FSDP 的参数分片相结合），也能轻松应对。
2. <b>支持 Eager 模式下的张量并行</b>：相比早期的 ShardedTensor，DTensor 允许更灵活地混合分片和复制操作，并在 Eager 模式下启用张量并行。
3. <b>编译器的基础构建块</b>：充当 SPMD（单程序多数据）编程模型的入口点和基于编译器的分布式训练的基础构建块。

## 具体代码示例
以下是一个在 PyTorch 中使用 DTensor 实现张量并行的标准代码示例。该示例展示了如何通过设备网格创建分片计划，并使用 parallelize_module 对模型进行自动化并行处理：

In [ ]:
from torch.distributed._tensor import DeviceMesh
from torch.distributed.tensor.parallel import PairwiseParallel, parallelize_module

# 1. 通过设备网格根据给定的 world_size 创建分片计划
device_mesh = DeviceMesh("cuda", torch.arange(0, args.world_size))

# 2. 初始化模型和优化器
model = ToyModel().cuda(rank)
LR = 0.25
optimizer = torch.optim.SGD(model.parameters(), lr=LR)

# 3. 根据给定的并行风格并行化模块
# 这里指定为 PairwiseParallel，将 colwise 和 rowwise 样式串联为固定对，
# 这与 Megatron-LM 的做法类似
model = parallelize_module(model, device_mesh, PairwiseParallel())

# 4. 执行常规的前向、后向传播及优化器更新
for i in range(args.iter_nums):
    # 对于张量并行(TP)，所有 TP rank 的输入需要相同
    # 设置随机种子是为了模仿数据加载器的行为
    if rank == 0:
        print(f"-----------{i}--------------")
    torch.manual_seed(i)
    
    inp = torch.rand(20, 10).cuda(rank)
    if rank == 0:
        print(f"rank: {rank} , input shape: {inp.shape}")
        
    output = model(inp)
    
    if rank == 0:
        print(f"rank: {rank} , output shape: {output.shape}")

- 在实际应用中（例如自动驾驶感知模型的 CNN 训练），DTensor 还可以被配置为将输入值和中间激活值按特定维度切分（Shard）到多个 GPU 上，
- 而将模型权重和优化器状态完整复制（Replicate）到各个 GPU 上。这种灵活的组合能够有效降低单个 GPU 的内存占用和带宽压力，极大提高大规模集群的训练效率。